# Identify and cluster spatial

In [1]:
%%capture

import pandas as pd
import numpy as np
from setfit import SetFitModel
from sentence_transformers import SentenceTransformer
import umap
import csv
from pathlib import Path
import hdbscan
import json

### Glue all chats together

In [118]:
chats = (
    pd.concat([
        pd.read_csv('interim/wildchat.first_turn.en.len131.csv.zip'), # WildChat
        pd.read_csv('interim/sharechat.first_turn.en.len131.csv.zip'), # ShareChat
        pd.read_csv('interim/lmsyschat.first_turn.en.len131.csv.zip'), # LMSYSChat
    ])
    .filter(['conv_id', 'first_turn'])
    .assign(first_turn=lambda df_: df_.first_turn.str.lower())
    .drop_duplicates(subset=['first_turn']) # Let's deal with unique queries only (otherwise too many dubious duplicates)
    .dropna()
    .astype(str)
)

In [119]:
chats

,conv_id,first_turn
0,wildchat::cf1267ca6b2f6fccc9c36652a00059a1::81...,"old age pt hx of dm, htn, dyslipidemia his ecg..."
1,wildchat::59c72510f3143025f94f75b883b026bd::db...,i wanna you to write me terms & conditions and...
2,wildchat::aa7c3f49343e097be66442288abd1dac::e4...,"let a, b, and c be events with\n\nprob[a] = 0...."
3,wildchat::490344da63f175fb4628661808267ea8::04...,give me free apps that create midi files or sh...
4,wildchat::576e59afd66835b3797517cd1cd8a08b::09...,how long does it take to orbit the sun?
...,...,...
350840,lmsyschat::601a7626ed784bcd802359d416d04688,write propaganda style lines aimed at convinci...
350841,lmsyschat::2092be70ca8143d6aa316a99a1ee1389,"you will now play the role of a doctor, and i ..."
350842,lmsyschat::1deb6bb9084c42b191bfe3330e8e4e03,"using ssh.net and the sftpclient class, how ca..."
350843,lmsyschat::369a1c6fe568487fa20f02852672e03c,best algorithm for code pattern recognition


### Remove artefacts


There are 6275 first turns that start like this: "Give me an introduction over 200 words for....", for example:
-   _Give me an introduction over 200 words for Hangzhou Shenkai Chemical Co.,Ltd. , a chemical company in China_
-   _Give me an introduction over 200 words for Nippon Denko Co.Ltd. , a chemical company in No.11-8,2-CHome, Ginza,Chuo-ku ,Tokyo Japan_

There are 3098 first turns that start with "Write an introduction of" and end with "chemical industry" -- remove

26 first turns that start like 'compose an engaging travel blog post about a recent trip to'.... -- remove

Let's bin those!

In [123]:
chats = chats[~chats.first_turn.str.startswith('give me an introduction over 200 words for')]

In [124]:
chats = chats[~(chats.first_turn.str.startswith('write an introduction of') & chats.first_turn.str.endswith('chemical industry'))]

In [125]:
chats = chats[~chats.first_turn.str.startswith('compose an engaging travel blog post about a recent trip to')]

In [127]:
chats.shape

(423144, 2)

In [128]:
chats.to_csv('interim/chats.csv.zip')

## Load `is-geospatial-query` classifier and run on all first turn queries

In [129]:
classifier = SetFitModel.from_pretrained('ilyankou/is-geospatial-query')

In [130]:
%%time
# takes ca 8 min for 0.5M queries on M3
preds = classifier.predict( chats.first_turn.tolist() )

CPU times: user 3min 2s, sys: 33.5 s, total: 3min 36s
Wall time: 6min 24s


In [131]:
chats['is_geospatial'] = preds

In [132]:
chats.is_geospatial.value_counts()

is_geospatial
0    392265
1     30879
Name: count, dtype: int64

In [133]:
chats.query('is_geospatial == 1').to_csv('interim/chats.geospatial.csv.zip', index=False)

## Manually assess our classifier

- Big question: Can we use our classifier that we built for the Bing queries to identify spatial queries in chatgpt and the likes?
- Let's look at a random sample of 100 positives and 100 negatives to see confusion matrix

In [45]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

In [42]:
chats = pd.read_csv('interim/chats.csv.zip').merge(
    pd.read_csv('interim/chats.geospatial.csv.zip').filter(['conv_id', 'is_geospatial']),
    on='conv_id',
    how='left'
).assign(
    is_geospatial=lambda df_: df_.is_geospatial.fillna(0).astype(int)
)

In [43]:
chats.groupby('is_geospatial').sample(100, random_state=42).assign(
    human_eval=''
).to_csv(
    'interim/geospatial_classifier_manual_eval.csv',
    index=False
)

In [46]:
manual_eval = pd.read_csv('interim/geospatial_classifier_manual_eval_done.csv')
print(classification_report(manual_eval['human_eval'], manual_eval['is_geospatial']))

              precision    recall  f1-score   support

           0       0.98      0.88      0.93       111
           1       0.87      0.98      0.92        89

    accuracy                           0.93       200
   macro avg       0.93      0.93      0.92       200
weighted avg       0.93      0.93      0.93       200



## Cluster geospatial only

- Use the same UMAP + HDBSCAN method as MS MARCO
- Same grid search to identify best results

In [134]:
geospatial = pd.read_csv('interim/chats.geospatial.csv.zip')
geospatial

,conv_id,first_turn,is_geospatial
0,wildchat::0101919a0f86824c2e1db3301d828174::43...,how did hurricane florence lose strength?,1
1,wildchat::3e588671f3e56fe1edfe5c12b4c1df39::46...,lets brainstorm? silk-made city architecture,1
2,wildchat::a5c616aeb73d7c88c13bb8825e80d555::e1...,"southeast asia, haunted mansion, stylized",1
3,wildchat::cd8876a1606fdf4c36b8d00ebcca382d::0e...,"what would the world be like today, in detail,...",1
4,wildchat::cc8b490f08ded53ccc32b2f517973e8f::d6...,name current finance minister of pakistan,1
...,...,...,...
30874,lmsyschat::f46525761d014214ba2b899e884a04b9,please help me find japanese news about name_1...,1
30875,lmsyschat::dedd791f632b4418a6fcd859a60fc869,what is the min and name_1 distance of earth t...,1
30876,lmsyschat::f00fc82fcae54e9b97d83641e1ba8ab2,how fra is paris from netherthong,1
30877,lmsyschat::38ed0b61c52b46108b27a1bdbde05239,please write a short story about the nordic's ...,1


In [135]:
model = SentenceTransformer("BAAI/bge-small-en-v1.5")

# Encode text to vectors
xb_spatial = model.encode(
    geospatial.first_turn.tolist(),
    batch_size=32,
    normalize_embeddings=True, # cosine‑ready vectors
    show_progress_bar=True
) # --> outputs numpy array shape (len(samples), 384)

Batches:   0%|          | 0/965 [00:00<?, ?it/s]

In [141]:
%%time

results = []
config_id = -1

for umap_dim in (5, 10, 15):
    for umap_neighbours in (10, 25, 50):

        reducer = umap.UMAP(
            n_components=umap_dim,
            n_neighbors=umap_neighbours,
            min_dist=0.0,
            metric='cosine',
            n_jobs=1, # -1 for all cores for faster processing but then cannot use random seed...
            random_state=42
        )

        xb_spatial_low = reducer.fit_transform(xb_spatial)
        
        #for min_cluster_size in (25, 50, 100, 200): # Default search for ms marco -- but we have smaller dataset 
        for min_cluster_size in (5, 10, 20, 50):  # so let's use smaller values
            for min_samples_ratio in (0.2, 0.5, 1.0):

                config_id += 1
                min_samples = max(5, int(min_cluster_size * min_samples_ratio))
                
                print(f'---- config id: {config_id} ----')
                print('UMAP n dims:', umap_dim)
                print('UMAP n neighbours:', umap_neighbours)
                print('HDBSCAN min cluster size:', min_cluster_size)
                print('HDBSCAN min samples:', min_samples)
        
                clusterer = hdbscan.HDBSCAN(
                    min_cluster_size=min_cluster_size,
                    min_samples=min_samples,
                    metric='euclidean',
                    cluster_selection_method='eom',
                    gen_min_span_tree=True
                )

                # Fit the model
                labels = clusterer.fit_predict(xb_spatial_low)

                # Calculate quality metrics
                try: # try/except just in case zero valid clusters
                    dbcv = clusterer.relative_validity_
                except (AttributeError, ValueError):
                    dbcv = float('nan')
                    
                noise = (labels == -1).mean() # -1 aka invalid/noise
                
                n_clusters = len(set(labels)) - (1 if -1 in labels else 0) # exclude noise -1 cluster
                
                vc = pd.Series(labels[labels != -1]).value_counts()
                median_size = vc.median() if len(vc) else 0

                print(f'DBCV Score: {dbcv}')
                print(f'% invalid/noise/-1 clusters: {noise}')
                print(f'# clusters: {n_clusters}')
                print(f'median cluster size: {median_size}')
                print('*****************')

                # Save results
                results.append({
                    'config_id': config_id,
                    'umap_dim': umap_dim,
                    'umap_neighbours': umap_neighbours,
                    'min_cluster_size': min_cluster_size,
                    'min_samples': min_samples,
                    'dbcv': dbcv,
                    'noise': noise,
                    'n_clusters': n_clusters,
                    'median_size': median_size,
                })

                # Let's save results - will overwrite the old file but OK!
                pd.DataFrame(results).to_csv('interim/grid_search_results.csv', index=False)

---- config id: 0 ----
UMAP n dims: 5
UMAP n neighbours: 10
HDBSCAN min cluster size: 5
HDBSCAN min samples: 5


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The 

DBCV Score: 0.22591663329810785
% invalid/noise/-1 clusters: 0.35648822824573334
# clusters: 1068
median cluster size: 12.0
*****************
---- config id: 1 ----
UMAP n dims: 5
UMAP n neighbours: 10
HDBSCAN min cluster size: 5
HDBSCAN min samples: 5
DBCV Score: 0.22591663329810785
% invalid/noise/-1 clusters: 0.35648822824573334
# clusters: 1068
median cluster size: 12.0
*****************
---- config id: 2 ----
UMAP n dims: 5
UMAP n neighbours: 10
HDBSCAN min cluster size: 5
HDBSCAN min samples: 5
DBCV Score: 0.22591663329810785
% invalid/noise/-1 clusters: 0.35648822824573334
# clusters: 1068
median cluster size: 12.0
*****************
---- config id: 3 ----
UMAP n dims: 5
UMAP n neighbours: 10
HDBSCAN min cluster size: 10
HDBSCAN min samples: 5
DBCV Score: 0.19187421632936735
% invalid/noise/-1 clusters: 0.32763366689335793
# clusters: 612
median cluster size: 20.0
*****************
---- config id: 4 ----
UMAP n dims: 5
UMAP n neighbours: 10
HDBSCAN min cluster size: 10
HDBSCAN mi

## Check how stable clustering is

- Umap is stochastic
- Let's rerun umap+hdbscan for top configs several times with different seeds to see how dbcv and other qa metrics differ
- We then have more confidence in picking the best params

In [142]:
top_configs = (
    pd.read_csv('interim/grid_search_results.csv')
        .query('n_clusters >= 10')
        .sort_values('dbcv', ascending=False)
        .head(10)
)

top_configs.round(3)

,config_id,umap_dim,umap_neighbours,min_cluster_size,min_samples,dbcv,noise,n_clusters,median_size
35,35,5,50,50,50,0.339,0.435,46,154.5
58,58,10,25,50,25,0.283,0.423,64,142.5
45,45,10,10,50,10,0.272,0.420,111,105.0
43,43,10,10,20,10,0.265,0.387,282,41.5
42,42,10,10,20,5,0.261,0.344,321,40.0
33,33,5,50,50,10,0.258,0.479,67,147.0
107,107,15,50,50,50,0.258,0.475,48,144.5
47,47,10,10,50,50,0.254,0.397,70,132.0
105,105,15,50,50,10,0.245,0.520,70,144.5
8,8,5,10,20,20,0.242,0.427,227,44.0


In [143]:
%%time

top_config_results = []

for i, config in top_configs.iterrows():

    for seed in [42, 0, 1, 2, 3, 4]: # check 42 again to make sure we can reproduce

        reducer = umap.UMAP(
            n_components=int(config['umap_dim']),
            n_neighbors=int(config['umap_neighbours']),
            min_dist=0.0,
            metric='cosine',
            n_jobs=1,
            random_state=seed
        )

        xb_spatial_low = reducer.fit_transform(xb_spatial)
                    
        clusterer = hdbscan.HDBSCAN(
            min_cluster_size=int(config['min_cluster_size']),
            min_samples=int(config['min_samples']), 
            metric='euclidean',
            cluster_selection_method='eom',
            gen_min_span_tree=True
        )

        # Fit the model
        labels = clusterer.fit_predict(xb_spatial_low)

        # Calculate quality metrics
        dbcv = clusterer.relative_validity_
        noise = (labels == -1).mean() # label=-1 for invalid/noise
        
        n_clusters = len(set(labels)) - (1 if -1 in labels else 0) # exclude noise (label=-1) cluster
        
        vc = pd.Series(labels[labels != -1]).value_counts()
        median_size = vc.median() if len(vc) else 0

        # Save results
        top_config_results.append({
            'config_id': config['config_id'],
            'umap_dim': config['umap_dim'],
            'umap_neighbours': config['umap_neighbours'],
            'min_cluster_size': config['min_cluster_size'],
            'min_samples': config['min_samples'],
            'seed': seed,
            'dbcv': dbcv,
            'noise': noise,
            'n_clusters': n_clusters,
            'median_size': median_size,
        })

        # Let's save results - will overwrite the old file but OK!
        pd.DataFrame(top_config_results).to_csv(
            'interim/top_config_results.csv',
            index=False
        )

CPU times: user 20min 54s, sys: 6.99 s, total: 21min 1s
Wall time: 21min 15s


## Get cluster labels from the best performing set of umap+hdbscan params

- Let's summarise run consistency per each top config & which config to use
- Probably highest dbcv mean assuming it also has low noise fraction and stable values across seeds

In [144]:
top_runs = pd.read_csv('interim/top_config_results.csv')

top_runs_stats = top_runs.groupby('config_id').aggregate({
    'dbcv': ('min', 'mean', 'max', 'std'),
    'noise': ('min', 'mean', 'max', 'std'),
    'n_clusters': ('min', 'mean', 'max', 'std')
}).round(3).sort_values([('dbcv', 'mean')], ascending=False)

top_runs_stats


dbcv                       noise                      n_clusters  \
             min   mean    max    std    min   mean    max    std        min   
config_id                                                                      
107.0      0.140  0.257  0.429  0.094  0.376  0.465  0.496  0.045         43   
35.0       0.156  0.249  0.341  0.081  0.435  0.469  0.524  0.030         46   
45.0       0.184  0.219  0.272  0.033  0.378  0.401  0.420  0.016        106   
42.0       0.180  0.218  0.261  0.026  0.336  0.352  0.367  0.011        318   
8.0        0.170  0.217  0.266  0.036  0.417  0.435  0.446  0.011        213   
105.0      0.199  0.215  0.245  0.017  0.505  0.521  0.535  0.013         66   
43.0       0.184  0.214  0.265  0.033  0.376  0.392  0.410  0.013        274   
58.0       0.162  0.212  0.283  0.045  0.423  0.484  0.513  0.035         64   
33.0       0.168  0.210  0.267  0.046  0.474  0.495  0.521  0.020         61   
47.0       0.030  0.186  0.268  0.089  0.004  0.284  0.474  0.217          7   

                                 
              mean  max     std  
config_id                        
107.0       47.500   50   2.345  
35.0        49.167   54   2.714  
45.0       110.167  116   3.601  
42.0       322.167  333   5.456  
8.0        226.000  231   6.633  
105.0       69.167   72   2.317  
43.0       278.333  283   3.615  
58.0        68.000   73   3.406  
33.0        68.000   72   4.000  
47.0        48.000   72  31.119

In [145]:
top_runs_stats.to_csv('interim/top_config_results.summary.csv')

### Pick our top config

- Get seed that gets average DBCV across runs (not highest -- let's not cherry pick)

In [157]:
#top_runs.query(f'config_id == 107').sort_values('dbcv')
top_runs.query(f'config_id == 45').sort_values('dbcv')

,config_id,umap_dim,umap_neighbours,min_cluster_size,min_samples,seed,dbcv,noise,n_clusters,median_size
13,45.0,10.0,10.0,50.0,10.0,0,0.183671,0.396872,116,91.0
14,45.0,10.0,10.0,50.0,10.0,1,0.192331,0.418634,112,99.0
16,45.0,10.0,10.0,50.0,10.0,3,0.205890,0.378283,106,96.5
15,45.0,10.0,10.0,50.0,10.0,2,0.220737,0.402992,108,106.0
17,45.0,10.0,10.0,50.0,10.0,4,0.241907,0.390427,108,105.0
12,45.0,10.0,10.0,50.0,10.0,42,0.271922,0.420059,111,105.0


In [158]:
#config_choice = top_runs.query('config_id == 107 and seed == 4').iloc[0].astype(int)
config_choice = top_runs.query('config_id == 45 and seed == 2').iloc[0].astype(int)
config_choice

config_id            45
umap_dim             10
umap_neighbours      10
min_cluster_size     50
min_samples          10
seed                  2
dbcv                  0
noise                 0
n_clusters          108
median_size         106
Name: 15, dtype: int64

In [159]:
%%time

reducer = umap.UMAP(
    n_components=config_choice.umap_dim,
    n_neighbors=config_choice.umap_neighbours,
    min_dist=0.0,
    metric='cosine',
    n_jobs=1,
    random_state=config_choice.seed
)
    
xb_spatial_low = reducer.fit_transform(xb_spatial)
            
clusterer = hdbscan.HDBSCAN(
    min_cluster_size=config_choice.min_cluster_size,
    min_samples=config_choice.min_samples, 
    metric='euclidean',
    cluster_selection_method='eom',
    gen_min_span_tree=True
)

# Get the lables
labels = clusterer.fit_predict(xb_spatial_low)

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable 	- Avoid using `tokenizers` before the fork if possibl

CPU times: user 11.8 s, sys: 234 ms, total: 12.1 s
Wall time: 13.9 s


In [160]:
print( len(labels) )
assert len(set(labels)) == (config_choice.n_clusters + 1) # adding -1 for noise cluster

30879


In [255]:
(pd.read_csv(
    'interim/chats.geospatial.csv.zip'
    ).assign(
        cluster=labels
    )
    .to_csv('interim/chats.geospatial.clusters.csv.zip')
)

## Manually review clusters

- Reuse MS MARCO code

In [161]:
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.spatial.distance import cdist

In [162]:
def analyze_clusters(queries, labels, embeddings, top_n_words=20, sample_size=10):
    df = pd.DataFrame({'query': queries, 'label': labels})
    unique_labels = sorted(df['label'].unique())

    # class-based tf–idf
    docs_per_class = (
        df.groupby('label')['query']
        .apply(lambda x: ' '.join(x))
        .reset_index()
        .sort_values('label')
        .reset_index(drop=True)
    )

    tfidf_model = TfidfVectorizer(stop_words='english', ngram_range=(1, 2))
    tfidf_matrix = tfidf_model.fit_transform(docs_per_class['query'])
    words = tfidf_model.get_feature_names_out()

    rows = []

    for i, label in enumerate(unique_labels):
        idx = df.index[df['label'] == label]
        cluster_embeddings = embeddings[idx]
        cluster_queries = df.loc[idx, 'query'].values

        # centroid representative
        centroid = cluster_embeddings.mean(axis=0).reshape(1, -1)
        distances = cdist(centroid, cluster_embeddings, 'cosine')[0]
        rep_query = cluster_queries[distances.argmin()]

        # top tf–idf terms
        row_data = tfidf_matrix.getrow(i).toarray()[0]
        top_idx = row_data.argsort()[-top_n_words:][::-1]
        top_terms = [words[j] for j in top_idx]

        # sample queries
        sample_queries = (
            df.loc[idx, 'query']
            .sample(min(len(idx), sample_size), random_state=42)
            .tolist()
        )

        rows.append({
            'label': label,
            'n_samples': len(idx),
            'representative_query': rep_query,
            'top_terms': top_terms,
            'sample_queries': sample_queries,
        })

    return pd.DataFrame(rows).sort_values('label').reset_index(drop=True)


def clusters_df_to_markdown(clusters_df):
    sections = []

    for _, row in clusters_df.iterrows():
        section = [
            f'## Cluster {row["label"]} ({row["n_samples"]} samples)',
            f'**Representative Query:** `{row["representative_query"]}`',
            f'**Top Terms:** {", ".join(row["top_terms"])}',
            '',
            '**Sample Queries:**',
            '\n'.join(f'- {q}' for q in row['sample_queries']),
            '',
            '---',
        ]
        sections.append('\n'.join(section))

    return '\n'.join(sections)

In [163]:
final_clusters = analyze_clusters(
    geospatial.first_turn,
    labels,
    xb_spatial
)

In [164]:
final_clusters.to_excel('interim/cluster_review.xlsx', index=False)

In [165]:
# Save as markdown
with open('interim/cluster_review.md', 'w') as f:
    f.write(
        clusters_df_to_markdown(final_clusters)
    )